# LC 295 — Find Median from Data Stream

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Maintain two heaps — a max-heap
for the lower half and a min-heap for the upper half — keeping
them balanced (sizes differ by at most 1). The median is always
at the tops of these heaps, available in O(1).
</div>

## Official Problem Statement

The **median** is the middle value in an ordered integer list.
If the size is even, the median is the mean of the two middle
values.

Implement `MedianFinder`:
- `MedianFinder()` initializes the object.
- `void addNum(int num)` adds integer `num` from the data stream.
- `double findMedian()` returns the median of all elements so far.

**Example:**
```
MedianFinder mf = MedianFinder()
mf.addNum(1)  →  lo=[-1]  hi=[]
mf.addNum(2)  →  lo=[-1]  hi=[2]
mf.findMedian() → 1.5
mf.addNum(3)  →  lo=[-2,-1]  hi=[3]
mf.findMedian() → 2.0
```
**Constraints:**
- `-10^5 <= num <= 10^5`
- At most `5 * 10^4` calls to `addNum` and `findMedian`.
- `findMedian` called at least once after every `addNum`.

## What This Is Actually Asking

We need the median after every insertion without re-sorting the
entire list. Sorting each time is O(n log n) per query — too slow.

The trick: split the sorted stream at the midpoint. The lower half
lives in a max-heap (so its largest is at the top) and the upper
half lives in a min-heap (so its smallest is at the top). The
median is either the top of the larger heap, or the average of
both tops when sizes are equal.

Python's `heapq` is a min-heap, so we negate values to simulate
a max-heap for the lower half.

## Walk Through an Example by Hand

```
addNum protocol:
  1. Always push to lo (max-heap, store negated)
  2. Move lo's top to hi (ensures lo's max <= hi's min)
  3. If len(hi) > len(lo): move hi's top back to lo

addNum(1):
  push -1 to lo  → lo=[-1]
  pop -1, push 1 to hi → lo=[], hi=[1]
  len(hi)>len(lo): pop 1, push -1 to lo → lo=[-1], hi=[]

addNum(2):
  push -2 to lo → lo=[-2,-1] (heap: -2 at top)
  pop -2, push 2 to hi → lo=[-1], hi=[2]
  len(hi)==len(lo): balanced  → median=(1+2)/2=1.5

addNum(3):
  push -3 to lo → lo=[-3,-1] (heap: -3 at top)
  pop -3, push 3 to hi → lo=[-1], hi=[2,3]
  len(hi)>len(lo): pop 2, push -2 to lo
  lo=[-2,-1], hi=[3] → median = -lo[0] = 2.0
```

## The Picture

```
Stream so far (sorted): [1, 2, 3, 4, 5]

          LOWER HALF          UPPER HALF
         (max-heap lo)       (min-heap hi)

         [ 1  2 ]            [ 3  4  5 ]
               ^               ^
           lo top=2        hi top=3

  sizes: len(lo)=2, len(hi)=3  →  median = hi[0] = 3

  Balanced (diff <= 1):
  ┌──────────────────────────────────┐
  │  lo (max-heap)  │  hi (min-heap) │
  │  [1, 2]  top→2  │  3 ←top [3,4,5]│
  └──────────────────────────────────┘
                    ↑
              median lives here

  Invariant: every value in lo <= every value in hi
  We enforce this by always routing through lo→hi.
```

## When To Use This Pattern

- When you need the **median** (or any order statistic) from a
  dynamic stream, think **two heaps**.
- When you need the top of two sorted halves in O(1), think
  **max-heap + min-heap partition**.
- When Python only provides min-heap, think
  **negate values** to simulate a max-heap.
- When a design problem asks for O(log n) insert and O(1) query,
  think **heap-based balanced split**.
- When sizes must stay balanced after each operation, think
  **rebalance by moving tops between heaps**.

## The Approach

Keep two heaps: `lo` (max-heap, lower half, values negated) and
`hi` (min-heap, upper half). On every `addNum`: push to `lo`,
then move `lo`'s top to `hi` (this enforces the ordering
invariant). If `hi` grows larger than `lo`, move `hi`'s top back
to `lo`.

After this protocol, `lo` has equal or one more element than `hi`.
For `findMedian`: if sizes are equal return `(-lo[0]+hi[0])/2`;
otherwise return `-lo[0]`.

In [ ]:
# Imports
import heapq

In [ ]:
# ----------------------------------------------------------
# Harness — op-replay style for design problem
# ----------------------------------------------------------
def test_harness(cls):
    cases = [
        {
            "ops":  ["addNum","addNum","findMedian",
                     "addNum","findMedian"],
            "args": [[1],[2],[],[3],[]],
            "expected": [None,None,1.5,None,2.0],
        },
        {
            "ops":  ["addNum","findMedian"],
            "args": [[5],[]],
            "expected": [None,5.0],
        },
        {
            "ops":  ["addNum","addNum","addNum",
                     "findMedian"],
            "args": [[6],[10],[2],[]],
            "expected": [None,None,None,6.0],
        },
    ]
    passed = 0
    for i, case in enumerate(cases):
        obj = cls()
        ok = True
        for op, arg, exp in zip(
            case["ops"], case["args"], case["expected"]
        ):
            res = getattr(obj, op)(*arg)
            if exp is not None and res != exp:
                print(f"FAILED case {i}: {op}({arg})"
                      f" expected={exp} got={res}")
                ok = False
        if ok:
            print(f"PASSED case {i}")
            passed += 1
    print(f"\n{passed}/{len(cases)} tests passed")

In [ ]:
class MedianFinder:
    """
    LC 295 — Find Median from Data Stream

    Two heaps:
      lo: max-heap (negate values) — lower half
      hi: min-heap — upper half

    addNum protocol:
      1. heappush(lo, -num)
      2. heappush(hi, -heappop(lo))  # enforce lo<=hi
      3. if len(hi)>len(lo): rebalance

    findMedian:
      equal sizes → (-lo[0]+hi[0])/2
      lo larger   → -lo[0]

    Time:  O(log n) addNum, O(1) findMedian
    Space: O(n)
    """
    def __init__(self):
        pass

    def addNum(self, num: int) -> None:
        pass
        # Debug: print(f"lo={self.lo} hi={self.hi}")

    def findMedian(self) -> float:
        pass

In [ ]:
# Uncomment and run when solution is ready
# test_harness(MedianFinder)

## Complexity

| Approach | addNum | findMedian | Space |
|---|---|---|---|
| Sort on each query | O(n log n) | O(1) | O(n) |
| Sorted list + bisect | O(n) | O(1) | O(n) |
| Two heaps (optimal) | O(log n) | O(1) | O(n) |
| Segment tree / BIT | O(log n) | O(log n) | O(n) |

## Real World Connection

**Citi / Finance context:** Real-time risk dashboards often
need running percentile metrics (median P&L, median latency)
over a live order stream. The two-heap pattern is exactly
what underpins online percentile tracking in monitoring systems.

In AWS CloudWatch or Datadog, streaming percentile approximations
(p50, p95, p99) are computed with similar heap-partitioning
ideas, though with bounded memory via digest structures.

For a data engineer ingesting millions of trade records, this
pattern avoids buffering and sorting entire windows — you always
have the current median in O(1) after each insert.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra